# SmartDocs — Single-Document RAG Chatbot

This notebook implements the SmartDocs MVP: upload **one PDF at a time**, retrieve relevant passages with FAISS, and generate grounded answers with Gemini.

**Pipeline:** PDF extraction → page-aware chunking → embeddings → FAISS retrieval → Gemini answer → source citations.


## 1. Install dependencies


In [1]:
!pip install -q sentence-transformers faiss-cpu pypdf flask python-dotenv "google-genai>=2.12.1,<3" "google-auth==2.49.0"

## 2. Import libraries and initialize models

In [2]:
import os
import numpy as np
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from google import genai
from google.colab import userdata

print("Core libraries imported successfully.")

Core libraries imported successfully.


In [3]:
# Load the embedding model
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [4]:
# Read the Gemini API key from Colab Secrets
gemini_api_key = userdata.get("GEMINI_API_KEY")

if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY was not found in Colab Secrets.")

gemini_client = genai.Client(api_key=gemini_api_key)

print("Gemini client initialized successfully.")

Gemini client initialized successfully.


## 3. Text chunking
Chunks are created separately for each PDF page so that page citations remain accurate.

In [12]:

def chunk_text(text, chunk_size=150, overlap=30):
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("overlap must be non-negative and smaller than chunk_size.")

    words = text.split()
    chunks = []
    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append({
            "chunk_id": len(chunks),
            "text": chunk
        })

        start += chunk_size - overlap

    return chunks


In [13]:
# Valid input
print(chunk_text("This is a sample document.", chunk_size=5, overlap=2))

# Invalid input
chunk_text("This is a sample document.", chunk_size=5, overlap=5)

[{'chunk_id': 0, 'text': 'This is a sample document.'}, {'chunk_id': 1, 'text': 'sample document.'}]


ValueError: overlap must be non-negative and smaller than chunk_size.

## 4. PDF processing and FAISS index
`process_pdf()` extracts text page by page, creates chunks with metadata, generates normalized embeddings, and builds a FAISS index.

In [7]:
from pypdf import PdfReader
import numpy as np
import faiss
import os

def process_pdf(
    pdf_path,
    chunk_size=100,
    overlap=20
):
    # Step 1: Get document name
    document_name = os.path.basename(pdf_path)

    # Step 2: Read PDF
    reader = PdfReader(pdf_path)

    document_chunks = []

    # Step 3: Extract text page by page
    for page_number, page in enumerate(reader.pages, start=1):

        page_text = page.extract_text() or ""

        if not page_text.strip():
            continue

        # Step 4: Chunk the extracted page text
        page_chunks = chunk_text(
            page_text,
            chunk_size=chunk_size,
            overlap=overlap
        )

        # Step 5: Attach metadata to each chunk
        for chunk in page_chunks:

            document_chunks.append({
                "chunk_id": len(document_chunks),
                "document_name": document_name,
                "page_number": page_number,
                "text": chunk["text"]
            })

    if not document_chunks:
        raise ValueError("No extractable text found in the PDF.")

    # Step 6: Generate embeddings
    texts = [
        chunk["text"] for chunk in document_chunks
    ]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True
    ).astype("float32")

    # Step 7: Normalize embeddings
    faiss.normalize_L2(embeddings)

    # Step 8: Create FAISS index
    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(embeddings)

    # Step 9: Return processed document
    return {
        "document_name": document_name,
        "chunks": document_chunks,
        "embeddings": embeddings,
        "index": index,
        "total_pages": len(reader.pages),
        "total_chunks": len(document_chunks)
    }

## 5. Question answering
`ask_smartdocs()` retrieves the most relevant chunks and asks Gemini to answer using only that context. The threshold of `0.30` is provisional and should be calibrated with more documents and questions.

In [8]:
def ask_smartdocs(
    question,
    processed_document,
    top_k=2,
    threshold=0.30
):

    # 1. Generate question embedding
    query_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    # 2. Normalize query embedding
    faiss.normalize_L2(query_embedding)

    # 3. Search the document's FAISS index
    scores, indices = processed_document["index"].search(
        query_embedding,
        top_k
    )

    # 4. Check relevance
    best_score = float(scores[0][0])

    if best_score < threshold:
        return {
            "question": question,
            "answer": "I couldn't find sufficiently relevant information in the document.",
            "sources": [],
            "similarity_scores": scores[0].tolist()
        }

    # 5. Retrieve matching chunks
    retrieved_chunks = []
    sources = []

    for score, idx in zip(scores[0], indices[0]):

        if idx == -1 or score < threshold:
            continue

        chunk = processed_document["chunks"][idx]

        retrieved_chunks.append(
            f"""
            Source: {chunk['document_name']}
            Page: {chunk['page_number']}

            {chunk['text']}
            """
        )

        sources.append({
            "document_name": chunk["document_name"],
            "page_number": chunk["page_number"],
            "chunk_id": chunk["chunk_id"],
            "similarity": float(score)
        })

    context = "\n\n".join(retrieved_chunks)

    # 6. Construct prompt
    prompt = f"""
    You are SmartDocs, a document question-answering assistant.

    Answer using ONLY the provided document context.

    Do not invent information.

    If the answer is unavailable, say that the
    document does not contain the answer.

    Cite the document name and page number.

    DOCUMENT CONTEXT:
    {context}

    USER QUESTION:
    {question}
    """

    # 7. Generate answer
    response = gemini_client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=prompt
    )

    return {
        "question": question,
        "answer": response.text,
        "sources": sources,
        "similarity_scores": scores[0].tolist()
    }

## 6. Upload one PDF
Upload exactly one PDF. A new document becomes active only after it has been processed successfully.

In [9]:
from google.colab import files

uploaded = files.upload()

# Accept exactly one PDF
if len(uploaded) != 1:
    raise ValueError("Please upload exactly one PDF file.")

pdf_filename = next(iter(uploaded))

if not pdf_filename.lower().endswith(".pdf"):
    raise ValueError("Only PDF files are supported.")

# Process the new PDF
new_document = process_pdf(pdf_filename)

# Make it the active document only after successful processing
active_document = new_document

print("Document uploaded and processed successfully!")
print("Active document:", active_document["document_name"])
print("Total pages:", active_document["total_pages"])
print("Total chunks:", active_document["total_chunks"])

Saving solar_energy_guide_smartdocs.pdf to solar_energy_guide_smartdocs.pdf
Document uploaded and processed successfully!
Active document: solar_energy_guide_smartdocs.pdf
Total pages: 3
Total chunks: 11


## 7. Test the active document
Start with one question, then run the evaluation set below.

In [10]:
result = ask_smartdocs(
    question="What does an inverter do in a solar energy system?",
    processed_document=active_document
)

print("Question:", result["question"])
print("\nAnswer:", result["answer"])
print("\nSources:")

for source in result["sources"]:
    print(
        f"Document: {source['document_name']} | "
        f"Page: {source['page_number']} | "
        f"Similarity: {source['similarity']:.4f}"
    )

Question: What does an inverter do in a solar energy system?

Answer: An inverter converts the direct-current (DC) electricity generated by the solar panels into alternating-current (AC) electricity used by household appliances or for grid export.

Source: solar_energy_guide_smartdocs.pdf, Page 1 & 2

Sources:
Document: solar_energy_guide_smartdocs.pdf | Page: 1 | Similarity: 0.6781
Document: solar_energy_guide_smartdocs.pdf | Page: 2 | Similarity: 0.6171


In [11]:
test_questions = [
    "What is the total illustrative system cost?",
    "How much does a battery cost?",
    "What is the inverter's purpose?",
    "How many days of written notice are required for membership cancellation?",
    "What is the cost of personal training?"
]

for question in test_questions:
    result = ask_smartdocs(
        question,
        active_document,
        top_k=2,
        threshold=0.30
    )

    print("\n" + "=" * 60)
    print("Question:", question)
    print("Answer:", result["answer"])
    print("Sources:", result["sources"])


Question: What is the total illustrative system cost?
Answer: The illustrative total in the budget example is $5,200.

Source: solar_energy_guide_smartdocs.pdf, Page: 3
Sources: [{'document_name': 'solar_energy_guide_smartdocs.pdf', 'page_number': 3, 'chunk_id': 8, 'similarity': 0.36868005990982056}, {'document_name': 'solar_energy_guide_smartdocs.pdf', 'page_number': 3, 'chunk_id': 10, 'similarity': 0.3615875244140625}]

Question: How much does a battery cost?
Answer: The document does not contain the answer.
Sources: [{'document_name': 'solar_energy_guide_smartdocs.pdf', 'page_number': 2, 'chunk_id': 6, 'similarity': 0.371110737323761}, {'document_name': 'solar_energy_guide_smartdocs.pdf', 'page_number': 3, 'chunk_id': 10, 'similarity': 0.3584842085838318}]

Question: What is the inverter's purpose?
Answer: Based on the provided document, the inverter converts DC (direct-current) electricity into AC (alternating-current) electricity, which is used by most household appliances (solar

## Current MVP scope and limitations

- One active PDF at a time; a successful new upload replaces the previous active document.
- Text-based PDFs are supported. Scanned PDFs require OCR, which is not implemented yet.
- FAISS uses exact inner-product search on normalized embeddings (equivalent to cosine similarity).
- The similarity threshold is provisional; scores are not confidence probabilities.
- Flask and frontend dependencies are installed for the next stage, but the API has not yet been implemented.